# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## Text as data: a `TextEncoder` class

In the LELA60331 seminars you have built **logistic regression** step by step, as a long series of notebook cells, and trained it on simulated data. In this week's and next week's LELA60331 seminars you will use it on real text: classifying Yelp reviews as positive or negative (**sentiment analysis**). In this workbook and the next you will build the same classifier as a **class**, using what you learned about classes in Week 7. By the end of Week 9 you will have a `LogisticRegression` class that can be used like this:

```python
model = LogisticRegression(vocab_size=5000, lr=0.4, n_epochs=500)
model.train(train_texts, train_labels)
print(model.evaluate(test_texts, test_labels))
```

This week we build the part that turns text into numbers. It prepares you for this week's LELA60331 seminar (*Turning text into features*), which does the same thing step by step.

| Part | Topic |
|---|---|
| 1 | From a script to a class |
| 2 | Loading the reviews |
| 3 | The `TextEncoder` class |
| 4 | Encoding the Yelp reviews |
| 5 | A different tokeniser, by inheritance |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · From a script to a class

In the LELA60331 seminars, each step of the classifier is a separate cell, and the cells share their results through **global names**. Even for the simulated data in LELA60331 Week 7 there are a lot of them:

| Name | What it holds |
|---|---|
| `x`, `y` | the features and the labels |
| `weights`, `bias` | the model's parameters |
| `lr`, `n_epochs` | the training settings |
| `q`, `errors`, `loss_history` | the probabilities, the errors and the losses |
| `score`, `sigmoid`, `average_loss`, `mat_vec`, ... | the functions that work on them |

Real text needs even more: a vocabulary, the encoded texts, separate training and test data, and so on.

This works, but it is fragile:

* Run the cells in the wrong order, or re-run one by mistake, and the names hold the wrong values. (In LELA60331 Week 7, `q` holds the probabilities from whichever cell you ran last: the untrained model, the trained one, or one with a different learning rate.)
* You can't easily have **two** models at once, say one trained with each learning rate, because they would both want the name `weights`.
* To use the classifier on new data, you have to remember which cells to copy, and in which order.

A class solves all three problems. It bundles the **data** (as instance variables) together with the **operations** on it (as methods), as in Week 7. Each model is then one object, with its own vocabulary and its own weights, and the rest of the program only needs to know what its methods do.

### The plan

We build the classifier in two layers, using **inheritance** (Week 7, Part 4):

* **This week:** `TextEncoder`, a class that knows how to turn texts into one-hot features
* **Next week:** `LogisticRegression`, a **subclass** of `TextEncoder` that adds training, prediction and evaluation. A logistic regression model *is a* text encoder that can also learn.

| `TextEncoder` | What it is or does |
|---|---|
| `_vocab_size` | instance variable: how many words to keep in the vocabulary |
| `_type_list` | instance variable: the vocabulary words, in column order |
| `_word_to_index` | instance variable: a dict from each vocabulary word to its column |
| `tokenise(text)` | method: split a text into a list of tokens |
| `fit_vocabulary(texts)` | method: build the vocabulary from a list of texts |
| `one_hot_encode(texts)` | method: turn a list of texts into sparse one-hot rows |
| `decode(row)` | method: turn a sparse row back into its words |
| `len(encoder)`, `encoder[j]` | special methods: the size of the vocabulary, and the word for column `j` |

### 🏋️ Exercise 1 · Practice — Data or method?

The full classifier will need each of the things below. For each one, say whether it should be stored as an `"instance variable"` or written as a `"method"`.

In [ ]:
design = {
    "the 5000 weights": ???,
    "calculate precision and recall": ???,
    "the learning rate": ???,
    "turn a list of texts into one-hot rows": ???,
    "the dict from words to columns": ???,
    "one step of gradient descent": ???,
}

In [ ]:
# ✅ Check
check("data", lambda: design["the 5000 weights"] == design["the learning rate"] == design["the dict from words to columns"] == "instance variable")
check("operations", lambda: design["calculate precision and recall"] == design["turn a list of texts into one-hot rows"] == design["one step of gradient descent"] == "method")

---
# Part 2 · Loading the reviews

The data set is the one you will use in this week's LELA60331 seminar: 10,000 reviews from the website Yelp, each labelled `negative` (1 or 2 stars) or `positive` (3 or 4 stars). The text has already been lower-cased, and its punctuation separated from the words by spaces, so that it is easy to split into words. A line beginning with `!` is a command for the computer, not Python: `wget` downloads the file.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

### 🏋️ Exercise 2 · Practice — `load_reviews(path)`

Write a function that reads the file and returns **two** lists (Week 4, *Returning several values*): the texts, and the labels as numbers, `1` for positive and `0` for negative.

The file is **tab-separated**: each line is a review, a tab (`"\t"`), then the label. The first line is a header, so skip it. Fill in:

1. the two fields of the line. *Hint:* `rstrip()` removes the newline at the end, and `split("\t")` splits at the tab.
2. adding the text to `texts`
3. adding the label to `labels`, as 1 or 0. *Hint:* `int(fields[1] == "positive")`

In [ ]:
def load_reviews(path):
    texts = []
    labels = []
    with open(path) as fp:
        lines = fp.readlines()
    for line in lines[1:]:              # skip the header line
        fields = ???          # 1. [text, label]
        ???                     # 2.
        ??? # 3.
    return texts, labels

texts, labels = load_reviews("yelp_reviews.txt")
print(len(texts), sum(labels))
print(labels[0], texts[0][:60])

In [ ]:
# ✅ Check
check("10000 reviews", lambda: len(texts) == 10000 and len(labels) == 10000)
check("5000 positive", lambda: sum(labels) == 5000 and set(labels) == {0, 1})
check("the first review", lambda: texts[0].startswith("all i can say") and labels[0] == 0)

We want to know how well a classifier does on reviews it has **never seen**, so we split the reviews: 8000 to train on, and 2000 to test on. `random.sample(sequence, k)` picks `k` different elements at random from `sequence`: here, the positions of the training reviews. The test reviews are all the others. With `random.seed(1)` this is exactly the split you will use in the LELA60331 seminar, so your numbers will match. Run the cell.

In [ ]:
import random

random.seed(1)
train_ints = random.sample(range(len(texts)), 8000)
train_set = set(train_ints)
test_ints = [i for i in range(len(texts)) if i not in train_set]

train_texts = [texts[i] for i in train_ints]
train_labels = [labels[i] for i in train_ints]
test_texts = [texts[i] for i in test_ints]
test_labels = [labels[i] for i in test_ints]
print(len(train_texts), len(test_texts), sum(train_labels), sum(test_labels))

---
# Part 3 · The `TextEncoder` class

## Turning text into features

Logistic regression needs every text to be a list of numbers, its **features**. In LELA60331 Week 7 each text had just two (the log counts of positive and of negative words). For real text we use one feature for every word in a **vocabulary**: the feature is 1 if the word occurs in the text and 0 if it doesn't. This is called **one-hot encoding**.

* The **vocabulary** is the list of words we use as features. We take the most frequent words in the corpus (5000 of them for the Yelp reviews). Each word has a fixed position in the list, its **column**.
* Each text becomes a **row** with one number per column. Here is a vocabulary of 4 words, and the row for the text *"good food good"*:

| column | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| word | good | bad | food | service |
| *"good food good"* | 1 | 0 | 1 | 0 |

"good" occurs twice, but its column still holds a 1: we only record whether a word is **present**.

With 5000 words, almost every number in a row is 0: a review contains only a small fraction of the vocabulary. Storing all those 0s would make 8000 reviews take 40 million numbers. So we use a **sparse** representation instead: for each text we store only the **list of columns that hold a 1**. The row above becomes `[0, 2]`.

## The class

Here is what each part of the class does. Read through all of it, then complete the class in Exercise 3.

**`__init__(self, vocab_size=5000)`.** `vocab_size=5000` gives the parameter a **default value**: if you write `TextEncoder()`, `vocab_size` is 5000, and if you write `TextEncoder(100)` or `TextEncoder(vocab_size=100)`, it is 100. The constructor stores it, and starts with an empty vocabulary: an empty list `_type_list` and an empty dict `_word_to_index`.

**`tokenise(self, text)`.** Splits a text into tokens at the spaces. This is done for you. It is a method of its own, rather than a `split()` inside the other methods, so that a subclass can change how *every* method tokenises by overriding just this one (Part 5).

**`fit_vocabulary(self, texts)`.** Builds the vocabulary from the most frequent words in `texts`:

* count every token with a counting dict (Week 3). Note that it calls **another method of the same object**, `self.tokenise(text)`
* make a list of `(count, word)` pairs and sort it from largest to smallest with `sorted(..., reverse=True)`. Python sorts pairs by their first element, so the most frequent words come first
* keep the words from the first `self._vocab_size` pairs in `self._type_list`
* fill in `self._word_to_index`, so that `self._word_to_index[word]` is the column of `word`

**`__len__(self)` and `__getitem__(self, j)`** (Week 7, Part 3). `len(encoder)` gives the number of words in the vocabulary, and `encoder[j]` the word for column `j`.

**`index(self, word)`.** The column of `word`, or `None` if it isn't in the vocabulary. Done for you.

**`one_hot_encode(self, texts)`.** For each text, its sparse row: the sorted list of the columns that hold a 1. Words that are not in the vocabulary are skipped. The columns are collected first in a **set** (Week 4), so that a word that occurs several times is stored only once.

**`decode(self, row)`.** The opposite of encoding: turns a sparse row (a list of columns) back into a list of words. *Hint:* the object's own `[]` works inside its methods too: `self[j]` is the word for column `j`.

### 🏋️ Exercise 3 · Practice — Complete `TextEncoder`

Fill in the blanks 1–6. The cell won't run until every `???` is replaced, so work through them in order. The tests underneath use a tiny corpus, so you can check the answers by hand.

⭐ **Challenge (optional):** add a `__str__` method, so that `print(encoder)` shows something like `TextEncoder with 4 words: good, food, was, the`.

In [ ]:
class TextEncoder:
    """Turn texts into sparse one-hot rows over a vocabulary of the most frequent words."""

    def __init__(self, vocab_size=5000):
        self._vocab_size = vocab_size
        self._type_list = ???                                  # 1. an empty vocabulary
        self._word_to_index = ???

    def tokenise(self, text):
        """Split a text into a list of tokens."""
        return text.split()

    def fit_vocabulary(self, texts):
        """Build the vocabulary from the most frequent tokens in texts."""
        counts = {}
        for text in texts:
            for word in self.tokenise(text):
                counts[word] = ???         # 2. count the word
        pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
        self._type_list = ???   # 3.
        self._word_to_index = {}
        for j in range(len(self._type_list)):
            ???        # 4. word -> column

    def __len__(self):
        return len(self._type_list)

    def __getitem__(self, j):
        return self._type_list[j]

    def index(self, word):
        """The column of word, or None if it is not in the vocabulary."""
        return self._word_to_index.get(word)

    def one_hot_encode(self, texts):
        """A list with one sorted list of columns per text."""
        M = []
        for text in texts:
            row = set()
            for word in self.tokenise(text):
                if ???:                # 5. only vocabulary words
                    row.add(self._word_to_index[word])
            M.append(sorted(row))
        return M

    def decode(self, row):
        """The words for the columns in a sparse row."""
        return ???                          # 6.

    # ⭐ Challenge: __str__
    # def __str__(self):
    #     return ???


tiny = ["the food was good", "the service was bad", "good food good service", "bad food"]
enc = TextEncoder(vocab_size=4)
enc.fit_vocabulary(tiny)
print(len(enc), [enc[j] for j in range(len(enc))])
print(enc.index("food"), enc.index("service"))

🔮 **Predict:** the vocabulary is `good, food, was, the` (columns 0 to 3). What will `enc.one_hot_encode(["good good food", "terrible service"])` give? Write down your answer, then run the cell.

In [ ]:
rows = enc.one_hot_encode(["good good food", "terrible service"])
print(rows)
print(enc.decode(rows[0]))

In [ ]:
# ✅ Check
def _enc_separate():
    a, b = TextEncoder(2), TextEncoder(2)
    a.fit_vocabulary(["x y"])
    return len(a) == 2 and len(b) == 0
check("1. a new encoder has an empty vocabulary", lambda: len(TextEncoder()) == 0 and TextEncoder()._vocab_size == 5000)
check("2–3. the 4 most frequent words", lambda: [enc[j] for j in range(len(enc))] == ["good", "food", "was", "the"])
check("4. word_to_index", lambda: enc.index("food") == 1 and enc.index("the") == 3 and enc.index("service") is None)
check("5. one_hot_encode", lambda: rows == [[0, 1], []])
check("5. each column once, in order", lambda: enc.one_hot_encode(["the was food the good"]) == [[0, 1, 2, 3]])
check("6. decode", lambda: enc.decode([0, 1]) == ["good", "food"] and enc.decode([]) == [])
check("each encoder has its own vocabulary", _enc_separate)
check("⭐ challenge: __str__", lambda: str(enc) == "TextEncoder with 4 words: good, food, was, the")

"service" occurs twice in the tiny corpus, as often as "was" and "the", but it didn't make the top 4. When counts are tied, sorting the `(count, word)` pairs from largest to smallest puts the words in reverse alphabetical order, so "service" lost out.

---
# Part 4 · Encoding the Yelp reviews

Now we use the class on the real data. In this week's LELA60331 seminar you will do the same thing step by step, over several cells.

### 🏋️ Exercise 4 · Practice — Using the encoder

1. Make `encoder`, a `TextEncoder` with a vocabulary of 5000 words, and build its vocabulary from **all** the reviews in `texts`. (This is what the LELA60331 seminar does. Strictly, it would be better to use only the training reviews: see the ⭐ question below.)
2. `M_train` and `M_test`: the one-hot rows of the training texts and of the test texts
3. `average_ones`: the average number of 1s in a row of `M_train`, that is, the average length of a sparse row. How sparse are the rows? (What proportion of the 5000 columns is that?)
4. `first_words`: the first 10 words of the first training review, according to the encoder. *Hint:* decode row 0 of `M_train`, then slice.

Encoding takes a few seconds.

In [ ]:
encoder = ???                       # 1.
???

M_train = ???    # 2.
M_test = ???

average_ones = ???     # 3.
first_words = ???                       # 4.

print(len(encoder), encoder[0], encoder[1])
print(len(M_train), len(M_test), round(average_ones, 1))
print(first_words)

In [ ]:
# ✅ Check
check("1. 5000 words, most frequent first", lambda: len(encoder) == 5000 and encoder[0] == "." and encoder[1] == "the")
check("2. M_train and M_test", lambda: len(M_train) == 8000 and len(M_test) == 2000)
check("2. the first training row", lambda: M_train[0][:10] == [0, 1, 3, 4, 5, 6, 7, 8, 10, 17])
check("3. average_ones", lambda: abs(average_ones - 78.5) < 0.01)
check("4. first_words", lambda: first_words == ['.', 'the', 'and', 'i', 'to', 'a', 'was', 'it', 'is', 'they'])

`first_words` doesn't look much like the start of a review. Why not? What has the one-hot encoding thrown away?

⭐ **Challenge (optional):** we built the vocabulary from all 10,000 reviews, including the test reviews. Why is this, strictly speaking, cheating? Make a second encoder whose vocabulary is built from `train_texts` only. How many of its 5000 words are different? *Hint:* make a set of each encoder's words.

In [ ]:
# ⭐ Challenge

---
# Part 5 · A different tokeniser, by inheritance

How we tokenise affects everything else: the vocabulary, and so the features. In a notebook written as a series of cells, trying a different tokenisation means going back, editing the tokenisation cell, and re-running everything below it, losing the old version. With a class, you can instead write a **subclass** that overrides `tokenise`, and keep both versions.

Very frequent function words like *the* and *and* take up many places in our vocabulary. In Week 7 you wrote a `StopwordVocabulary` that ignores **stop words**. Now we do the same for the encoder.

### 🏋️ Exercise 5 · Practice — `StopwordEncoder`

Complete `StopwordEncoder`, a subclass of `TextEncoder`:

1. the constructor calls the parent's constructor with `vocab_size` (so that the instance variables are created), then stores the stop words in a **set** in `self._stopwords`
2. `tokenise(text)` calls the **parent's** `tokenise` with `super()`, then keeps only the tokens that are not stop words. *Hint:* a list comprehension with an `if`.

You don't need to write `fit_vocabulary` or `one_hot_encode` again.

In [ ]:
class StopwordEncoder(TextEncoder):
    """A TextEncoder that ignores stop words."""

    def __init__(self, stopwords, vocab_size=5000):
        ???                   # 1.
        self._stopwords = ???

    def tokenise(self, text):
        return ???   # 2.

stop = StopwordEncoder(["the", "a", "and", "was"], vocab_size=4)
print(stop.tokenise("the food was good and the service was bad"))
stop.fit_vocabulary(tiny)
print([stop[j] for j in range(len(stop))])

In [ ]:
# ✅ Check
check("1. still a TextEncoder", lambda: isinstance(stop, TextEncoder) and stop._vocab_size == 4)
check("2. tokenise drops stop words", lambda: stop.tokenise("the food was good and the service was bad") == ["food", "good", "service", "bad"])
check("fit_vocabulary uses the new tokenise", lambda: stop.index("the") is None and stop.index("service") is not None)
check("one_hot_encode uses the new tokenise", lambda: stop.one_hot_encode(["the bad food"]) == [sorted([stop.index("bad"), stop.index("food")])])

🔮 **Predict:** `fit_vocabulary` and `one_hot_encode` belong to `TextEncoder`, and you didn't change them. Yet they now skip the stop words. Why? (Think about Week 7, Exercise 5: when `fit_vocabulary` calls `self.tokenise(text)`, which class's version runs, and why?)

Finally, here is a stop-word encoder for the Yelp reviews. Run the cell. Which words have taken the places of the stop words?

In [ ]:
yelp_stop = StopwordEncoder([".", ",", "the", "and", "i", "to", "a", "was", "it", "of"])
yelp_stop.fit_vocabulary(texts)
print([yelp_stop[j] for j in range(15)])
print(yelp_stop.index("the"), encoder.index("the"))     # each encoder has its own vocabulary

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how a class helps
* design a class by deciding what is **data** (instance variables) and what is an **operation** (methods)
* give a parameter a **default value**
* write a `TextEncoder` class that builds a vocabulary and one-hot encodes texts, with `len()` and `[]` support
* call one method from another with `self.`, and override a single method in a subclass to change the behaviour of the others

In this week's LELA60331 seminar you will build the same vocabulary and one-hot rows step by step. Next week, in Week 9, you will write `LogisticRegression`, a subclass of `TextEncoder` that trains, predicts and evaluates: the whole review classifier in one object.